In [ ]:
import torch
import torch.nn as nn 
import torch.nn.functional as F 
from torch.utils.data import Dataset, DataLoader 
import joblib 
from tqdm import tqdm
import os 
import csv


class BlobDataset(Dataset):
    def __init__(self, noisy_file, clean_file, device):
        print(f"Loading {noisy_file}...")
        # Load datasets from disk
        noisy_dat = joblib.load(noisy_file)
        clean_dat = joblib.load(clean_file)

        # Pre-load to Device (MPS/Mac) to eliminate I/O bottlenecks during training
        self.noisy = torch.from_numpy(noisy_dat).float().reshape(-1, 1, 20, 20).to(device)
        self.clean = torch.from_numpy(clean_dat["blobs"]).float().reshape(-1, 1, 20, 20).to(device)
        
        # Combine Integral and Centroids into one target vector like the DDP script
        # Shape: [N, 3] -> (Integral, Centroid_X, Centroid_Y)
        self.targets = torch.cat([
            torch.from_numpy(clean_dat["integrals"]).float().unsqueeze(1),
            torch.from_numpy(clean_dat["cents"]).float()
        ], dim=1).to(device)

    def __len__(self):
        return len(self.noisy)

    def __getitem__(self, idx):
        return self.noisy[idx], self.clean[idx], self.targets[idx]

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()

        def conv_block (in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True)
            )
    
        self.enc1 = conv_block(in_channels, 32)
        self.enc2 = conv_block(32, 64)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = conv_block(64, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = conv_block(128,64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = conv_block(64, 32)

        self.final_conv = nn.Conv2d(32, out_channels, kernel_size=1)
        
        self.gap = nn.AdaptiveAvgPool2d(1) 
        self.reg_head = nn.Sequential( 
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3), 
            nn.Linear(64, 3) 
        )
    

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))

        b = self.bottleneck(self.pool(e2))

        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        clean_img = torch.sigmoid(self.final_conv(d1))

        reg_flat = self.gap(b).view(b.size(0), -1)
        reg_out = self.reg_head(reg_flat)

        return clean_img, reg_out



DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

NOISE_AMP = 0.0005 
NOISE_OFFSET = 1.0
WEIGHTS = torch.tensor([10.0, 1.0, 1.0]).to(DEVICE)


def weighted_mse_loss(input, target, weights):

    return ((input - target)**2 * weights).mean()


train_ds = BlobDataset("TRA_NOISY_DAT.joblib", "TRA_CLEAN_DAT.joblib", DEVICE)
val_ds = BlobDataset("VAL_NOISY_DAT.joblib", "VAL_CLEAN_DAT.joblib", DEVICE)
train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False)

model = UNet().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
criterion_img = nn.MSELoss()

best_val = float('inf')

for epoch in range(15):
    model.train()
    train_loss = 0.0
    
    t_img_loss, t_reg_loss = 0.0, 0.0
    
    for noisy, clean, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
        optimizer.zero_grad(set_to_none=True)
        

        noise = NOISE_AMP * (2.0 * torch.rand(noisy.shape, device=DEVICE) - 1.0)
        offset = NOISE_OFFSET * torch.rand((noisy.shape[0], 1, 1, 1), device=DEVICE)
        inputs = noisy + noise + offset
        
        p_img, p_reg = model(inputs)
        
       
        loss_i = criterion_img(p_img, clean)
        loss_r = weighted_mse_loss(p_reg, targets, WEIGHTS)
        
       
        loss = (loss_i * 20.0) + loss_r
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        t_img_loss += loss_i.item()
        t_reg_loss += loss_r.item()


    model.eval()
    val_loss = 0.0
    v_img_loss, v_reg_loss = 0.0, 0.0
    
    with torch.no_grad():
        for noisy, clean, targets in val_loader:
            p_img, p_reg = model(noisy)
            
            loss_i = criterion_img(p_img, clean)
            loss_r = weighted_mse_loss(p_reg, targets, WEIGHTS)
            
            val_loss += ((loss_i * 20.0) + loss_r).item()
            v_img_loss += loss_i.item()
            v_reg_loss += loss_r.item()

   
    avg_train = train_loss / len(train_loader)
    avg_val = val_loss / len(val_loader)
    
    scheduler.step(avg_val)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"\n--- Epoch {epoch+1:02d} ---")
    print(f"OVERALL: Train Loss: {avg_train:.6f} | Val Loss: {avg_val:.6f}")
    print(f"IMAGE  : Train MSE:  {t_img_loss/len(train_loader):.6f} | Val MSE: {v_img_loss/len(val_loader):.6f}")
    print(f"REGRESS: Train WMSE: {t_reg_loss/len(train_loader):.6f} | Val WMSE: {v_reg_loss/len(val_loader):.6f}")
    print(f"STATS  : Learning Rate: {current_lr:.2e}")

    if avg_val < best_val:
        best_val = avg_val
        torch.save(model.state_dict(), "best_merged_model_unet.pt")
        print(">> New Best Model Saved!")
